In [1]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Connect to the local MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("AQI_Baseline_Experiment")

<Experiment: artifact_location='/Users/MAC/Documents/research/mlops-test/mlops-from-scratch/mlruns/1', creation_time=1776087823104, experiment_id='1', last_update_time=1776087823104, lifecycle_stage='active', name='AQI_Baseline_Experiment', tags={}, trace_location=None, workspace='default'>

In [4]:
# Load data (the path assumes your notebook is in the notebooks/ folder)
df = pd.read_csv("../data/pollution.csv")

# Combine year, month, day, hour into a single datetime index
df["date"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df.set_index("date")

# Drop the original date columns and the row number ('No')
df = df.drop(columns=["No", "year", "month", "day", "hour"])
df.columns = [
    "pm2.5",
    "dew_point",
    "temp",
    "pressure",
    "wind_dir",
    "wind_speed",
    "snow",
    "rain",
]

# Time series data often uses forward-filling to handle missing sensor readings
df["pm2.5"] = df["pm2.5"].ffill()
df = df.dropna()  # Drop the very first row if it was NA and couldn't be forward-filled

display(df.head())

,pm2.5,dew_point,temp,pressure,wind_dir,wind_speed,snow,rain
date,,,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,SE,1.79,0,0
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,SE,2.68,0,0
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,SE,3.57,0,0
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,SE,5.36,1,0
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,SE,6.25,2,0


In [5]:
# Convert categorical wind direction into numerical columns
df = pd.get_dummies(df, columns=["wind_dir"], drop_first=True)

# Separate features (X) and target (y)
X = df.drop(columns=["pm2.5"])
y = df["pm2.5"]

# Chronological split: First 80% of time for training, last 20% for testing
split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Testing shapes: X={X_test.shape}, y={y_test.shape}")

Training shapes: X=(35040, 9), y=(35040,)
Testing shapes: X=(8760, 9), y=(8760,)


In [6]:
# Start an MLflow run
with mlflow.start_run(run_name="RandomForest_Baseline"):
    # 1. Define hyperparameters
    n_estimators = 50
    max_depth = 10

    # 2. Log parameters to MLflow
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)

    # 3. Train the model
    model = RandomForestRegressor(
        n_estimators=n_estimators, max_depth=max_depth, random_state=42
    )
    model.fit(X_train, y_train)

    # 4. Evaluate the model
    predictions = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)

    print(f"Model RMSE: {rmse:.2f}")
    print(f"Model MAE: {mae:.2f}")

    # 5. Log metrics and the model artifact to MLflow
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)

    # This saves the model weight files into your mlruns/ folder
    mlflow.sklearn.log_model(model, "model")

2026/04/13 14:56:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 14:56:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Model RMSE: 78.21
Model MAE: 53.94


2026/04/13 14:56:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run RandomForest_Baseline at: http://127.0.0.1:5000/#/experiments/1/runs/94785d4e5fc64e66b5c8d4cd2227ad6c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
